# 05 — Error Analysis

## 1. Objective

The goal of this notebook is to understand where the current best model makes mistakes and identify patterns that may help improve model performance.

We focus on:

- False Negatives: actual `Yes`, predicted `No`
- False Positives: actual `No`, predicted `Yes`
- Error patterns across important features
- Possible ideas for model improvement

In [1]:
import sys
sys.path.append("..")

from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.data_split import load_and_split_data
from src.preprocessing import build_tree_preprocessor

X_train, X_valid, y_train, y_valid = load_and_split_data()

xgb_model = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_valid)

## 3. Build the Error Table

To analyze model mistakes, we combine the validation features with:

- the actual target value;
- the model prediction.

This allows us to identify which observations are True Positives, True Negatives, False Positives, and False Negatives.

In [2]:
error_df = X_valid.copy()

error_df["actual"] = y_valid
error_df["predicted"] = xgb_pred

display(error_df.head())

,country,year,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type,actual,predicted
69,Kenya,2018,Urban,Yes,1,30,Male,Head of Household,Single/Never Married,Secondary education,Formally employed Private,1,1
13395,Rwanda,2016,Urban,Yes,4,30,Female,Spouse,Married/Living together,Primary education,Remittance Dependent,0,0
18888,Tanzania,2017,Urban,Yes,1,34,Female,Head of Household,Married/Living together,No formal education,Self employed,0,0
4660,Kenya,2018,Rural,No,7,45,Female,Spouse,Married/Living together,Primary education,Informally employed,0,0
13090,Rwanda,2016,Rural,Yes,5,35,Female,Spouse,Married/Living together,Primary education,Farming and Fishing,0,0


## 4. Label Prediction Types

Each validation observation is labeled as:

- True Positive (`TP`)
- True Negative (`TN`)
- False Positive (`FP`)
- False Negative (`FN`)

This makes it easier to isolate and study different types of model errors.

In [3]:
def get_prediction_type(row):
    if row["actual"] == 1 and row["predicted"] == 1:
        return "TP"
    elif row["actual"] == 0 and row["predicted"] == 0:
        return "TN"
    elif row["actual"] == 0 and row["predicted"] == 1:
        return "FP"
    else:
        return "FN"


error_df["prediction_type"] = error_df.apply(
    get_prediction_type,
    axis=1
)

print(error_df["prediction_type"].value_counts())

prediction_type
TN    3960
FN     426
TP     236
FP      83
Name: count, dtype: int64


## 5. Analyze False Negatives

For each important feature, we calculate the percentage of actual positive cases (`bank_account = 1`) that were incorrectly predicted as negative.

This helps identify groups where the model misses positive cases more often.

In [6]:
features_to_check = [
    "country",
    "cellphone_access",
    "education_level",
    "job_type"
]

positive_cases = error_df[error_df["actual"] == 1]

for col in features_to_check:
    print(f"\n--- {col} ---")

    fn_rate = (
        positive_cases
        .groupby(col)["prediction_type"]
        .apply(lambda x: (x == "FN").mean() * 100)
        .sort_values(ascending=False)
    )

    print(fn_rate.round(2))


--- country ---
country
Rwanda      78.46
Tanzania    60.68
Uganda      58.33
Kenya       57.64
Name: prediction_type, dtype: float64

--- cellphone_access ---
cellphone_access
No     100.00
Yes     63.07
Name: prediction_type, dtype: float64

--- education_level ---
education_level
No formal education                100.00
Primary education                   98.06
Secondary education                 70.79
Other/Dont know/RTA                 33.33
Vocational/Specialised training     26.44
Tertiary education                  18.32
Name: prediction_type, dtype: float64

--- job_type ---
job_type
Dont Know/Refuse to answer      100.00
No Income                       100.00
Informally employed              92.94
Government Dependent             92.86
Farming and Fishing              89.68
Remittance Dependent             88.46
Self employed                    71.60
Other Income                     69.70
Formally employed Private        22.13
Formally employed Government      6.35
Name: pr

### False Negative Findings

The model misses a large proportion of positive cases in several groups.

- Rwanda has the highest country-level false negative rate.
- Positive cases without cellphone access are especially difficult to identify.
- False negative rates are very high for respondents with no formal or primary education.
- Informal employment, farming/fishing, and several low-income job groups also show high false negative rates.
- Positive cases with tertiary education or formal government employment are identified much more successfully.

## 6. Analyze False Positives

For each important feature, we calculate the percentage of actual negative cases (`bank_account = 0`) that were incorrectly predicted as positive.

This helps identify groups where the model produces false positive predictions more often.

In [7]:
negative_cases = error_df[error_df["actual"] == 0]

for col in features_to_check:
    print(f"\n--- {col} ---")

    fp_rate = (
        negative_cases
        .groupby(col)["prediction_type"]
        .apply(lambda x: (x == "FP").mean() * 100)
        .sort_values(ascending=False)
    )

    print(fp_rate.round(2))


--- country ---
country
Kenya       5.03
Rwanda      1.55
Uganda      0.92
Tanzania    0.86
Name: prediction_type, dtype: float64

--- cellphone_access ---
cellphone_access
Yes    2.91
No     0.00
Name: prediction_type, dtype: float64

--- education_level ---
education_level
Vocational/Specialised training    37.50
Tertiary education                 19.30
Other/Dont know/RTA                18.18
Secondary education                 4.22
Primary education                   0.17
No formal education                 0.00
Name: prediction_type, dtype: float64

--- job_type ---
job_type
Formally employed Government    75.00
Formally employed Private       27.17
Other Income                     3.37
Self employed                    2.14
Informally employed              0.58
Farming and Fishing              0.51
Remittance Dependent             0.46
Dont Know/Refuse to answer       0.00
Government Dependent             0.00
No Income                        0.00
Name: prediction_type, dtype: fl

### False Positive Findings

- Kenya has the highest country-level false positive rate.
- False positives are more common among respondents with cellphone access.
- Higher education levels show noticeably higher false positive rates.
- Formal employment groups also show high false positive rates.
- Overall, the model appears more likely to predict a bank account for respondents with stronger socioeconomic indicators.

## 7. Key Findings and Improvement Ideas

### Key Findings

- The model misses many positive cases in Rwanda.
- False negative rates are especially high for respondents with lower education levels and informal or low-income job types.
- The model performs much better on positive cases with tertiary education and formal employment.
- False positives are more common among respondents with higher education and formal employment.
- This suggests that the model may rely strongly on socioeconomic indicators when predicting bank account ownership.

### Improvement Ideas

Based on the error patterns, the next experiments will focus on:

- class weighting to improve minority-class detection;
- probability threshold adjustment;
- hyperparameter tuning;
- comparing the improved models using Macro F1 and Class 1 Recall.